# RetailFlow – Data Quality Analysis
Phase 2 of RetailFlow Lakehouse development.
Evaluates data completeness, schema validity, referential integrity, and anomaly detection across all Olist e-commerce datasets.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root
current = Path.cwd()
for candidate in [current, current / 'RetailFlow', *current.parents]:
    if (candidate / 'data' / 'raw').exists():
        ROOT = candidate
        break
    if (candidate / 'RetailFlow' / 'data' / 'raw').exists():
        ROOT = candidate / 'RetailFlow'
        break

RAW_PATH = ROOT / 'data' / 'raw'
print('RetailFlow Data Quality Engine Initialized')
print('Raw Data Path:', RAW_PATH)


In [ ]:
# Load core datasets for quality audit
orders = pd.read_csv(RAW_PATH / 'olist_orders_dataset.csv')
customers = pd.read_csv(RAW_PATH / 'olist_customers_dataset.csv')
payments = pd.read_csv(RAW_PATH / 'olist_order_payments_dataset.csv')
items = pd.read_csv(RAW_PATH / 'olist_order_items_dataset.csv')
products = pd.read_csv(RAW_PATH / 'olist_products_dataset.csv')
sellers = pd.read_csv(RAW_PATH / 'olist_sellers_dataset.csv')

raw_tables = {
    'orders': orders,
    'customers': customers,
    'payments': payments,
    'order_items': items,
    'products': products,
    'sellers': sellers
}

print(f'Successfully loaded {len(raw_tables)} datasets for quality audit.')


In [ ]:
# 1. Completeness & Null Analysis
quality_summary = []
for name, df in raw_tables.items():
    total_cells = df.shape[0] * df.shape[1]
    null_cells = df.isnull().sum().sum()
    completeness_pct = round((1 - (null_cells / total_cells)) * 100, 2)
    duplicates = df.duplicated().sum()
    quality_summary.append({
        'Dataset': name,
        'Rows': len(df),
        'Columns': df.shape[1],
        'Duplicates': duplicates,
        'Total Missing Values': null_cells,
        'Completeness %': completeness_pct
    })

df_quality = pd.DataFrame(quality_summary)
try:
    display(df_quality)
except Exception:
    print(df_quality.to_string(index=False))


In [ ]:
# 2. Referential Integrity Audit
print('=== Referential Integrity Audit ===')

orphan_cust = (~orders['customer_id'].isin(customers['customer_id'])).sum()
orphan_payments = (~payments['order_id'].isin(orders['order_id'])).sum()
orphan_items = (~items['order_id'].isin(orders['order_id'])).sum()
orphan_products = (~items['product_id'].isin(products['product_id'])).sum()
orphan_sellers = (~items['seller_id'].isin(sellers['seller_id'])).sum()

print(f'Orders without valid Customer: {orphan_cust}')
print(f'Payments without valid Order:    {orphan_payments}')
print(f'Items without valid Order:       {orphan_items}')
print(f'Items without valid Product:     {orphan_products}')
print(f'Items without valid Seller:      {orphan_sellers}')

if sum([orphan_cust, orphan_payments, orphan_items, orphan_products, orphan_sellers]) == 0:
    print('\nPASS: 100% Referential Integrity across all primary/foreign key relationships!')


In [ ]:
# 3. Business Logic & Anomaly Checks
negative_payments = (payments['payment_value'] < 0).sum()
zero_payments = (payments['payment_value'] == 0).sum()
negative_prices = (items['price'] <= 0).sum()
negative_freight = (items['freight_value'] < 0).sum()

print('=== Business Logic & Value Validation ===')
print(f'Negative Payments: {negative_payments}')
print(f'Zero-value Payments: {zero_payments}')
print(f'Non-positive Item Prices: {negative_prices}')
print(f'Negative Freight Costs: {negative_freight}')
print('\nData Quality Analysis Completed!')
